# Semiconductor Dataset Cleaning and Preprocessing

This notebook cleans `semiconductor_model_data_43_features.csv`, validates the exact 43-feature structure, preserves the requested date range, prepares the supervised training dataset, and defines leakage-safe preprocessing pipelines for Linear Regression, Random Forest, and XGBoost.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_FILE = Path(
    '/content/drive/MyDrive/'
    'Semiconductor-Stock-Return-Prediction-ML-Project/processed_data/'
    'semiconductor_model_data_43_features.csv'
)

OUTPUT_FILE = Path(
    '/content/drive/MyDrive/'
    'Semiconductor-Stock-Return-Prediction-ML-Project/processed_data/'
    'semiconductor_model_data_43_features_cleaned.csv'
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print('Input file:', INPUT_FILE)
print('Output file:', OUTPUT_FILE)


Input file: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/processed_data2/semiconductor_model_data_43_features.csv
Output file: /content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/processed_data2/semiconductor_model_data_43_features_cleaned.csv


## Exact 43-feature definition

In [ ]:
FEATURE_COLUMNS = [
    'open', 'high', 'low', 'adjusted_close', 'volume', 'daily_return',
    'sma_20', 'sma_50', 'ema_12', 'ema_26', 'momentum_20', 'distance_from_sma_20',
    'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'return_5d',
    'volatility_20', 'bollinger_width', 'atr_14', 'relative_volume',
    'revenue', 'eps', 'gross_margin', 'rd_expense', 'capex',
    'free_cash_flow', 'debt_to_equity', 'inventory',
    'fed_funds_rate', 'treasury_10y', 'vix', 'dxy', 'copper',
    'yield_curve_10y_2y',
    'sp500_return', 'nasdaq_return', 'soxx_return', 'qqq_return',
    'previous_day_return', 'previous_5_day_return',
    'previous_20_day_return', 'previous_60_day_return',
]

TARGET_COLUMNS = [
    'target_5_day_return',
    'target_10_day_return',
    'target_next_month_return'
]

# Select one target only when a later modeling or filtering step needs it.
TRAINING_TARGET = 'target_next_month_return'

FUNDAMENTAL_COLUMNS = [
    'revenue', 'eps', 'gross_margin', 'rd_expense',
    'capex', 'free_cash_flow', 'debt_to_equity', 'inventory',
]

assert len(FEATURE_COLUMNS) == 43
print('Feature count:', len(FEATURE_COLUMNS))


Feature count: 43


## Load and inspect the dataset

In [ ]:
df = pd.read_csv(INPUT_FILE)

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['filed_date'] = pd.to_datetime(df['filed_date'], errors='coerce')
df['period_end'] = pd.to_datetime(df['period_end'], errors='coerce')

df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print('Original shape:', df.shape)
print('\nOriginal missing values:')
display(df[FUNDAMENTAL_COLUMNS + TARGET_COLUMNS].isna().sum().to_frame('missing'))


Original shape: (30144, 50)

Original missing values:


,missing
revenue,0
eps,0
gross_margin,0
rd_expense,13
capex,684
free_cash_flow,684
debt_to_equity,0
inventory,0
target_5_day_return,0
target_10_day_return,0


## Remove duplicates and replace infinite values

In [ ]:
duplicate_count = df.duplicated(subset=['date', 'ticker']).sum()
print('Duplicate (date, ticker) rows:', duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates(subset=['date', 'ticker'], keep='last')

df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

numeric_columns = df.select_dtypes(include=[np.number]).columns
df[numeric_columns] = df[numeric_columns].replace([np.inf, -np.inf], np.nan)


Duplicate (date, ticker) rows: 0


## Fill missing fundamental values without using future information

The process first forward-fills within each ticker, then uses the same-date cross-sectional median, then a historical expanding median. The target is never filled.

In [ ]:
for column in FUNDAMENTAL_COLUMNS:
    df[column] = df.groupby('ticker')[column].ffill()

    same_date_median = df.groupby('date')[column].transform('median')
    df[column] = df[column].fillna(same_date_median)

    historical_median = (
        df[column]
        .shift(1)
        .expanding(min_periods=1)
        .median()
    )
    df[column] = df[column].fillna(historical_median)
    df[column] = df[column].fillna(0.0)

print('Missing values after fundamental cleanup:')
display(df[FUNDAMENTAL_COLUMNS + TARGET_COLUMNS].isna().sum().to_frame('missing'))


Missing values after fundamental cleanup:


,missing
revenue,0
eps,0
gross_margin,0
rd_expense,0
capex,0
free_cash_flow,0
debt_to_equity,0
inventory,0
target_5_day_return,0
target_10_day_return,0


## Inspect extreme debt-to-equity observations

In [ ]:
extreme_debt_rows = df.loc[
    df['debt_to_equity'].abs() > 20,
    ['date', 'ticker', 'debt_to_equity', 'filed_date', 'period_end'],
].sort_values('debt_to_equity')

print('Extreme debt-to-equity observations:', len(extreme_debt_rows))
display(extreme_debt_rows.head(20))


Extreme debt-to-equity observations: 47


,date,ticker,debt_to_equity,filed_date,period_end
10276,2017-05-30,MCHP,-205.207678,2017-05-30,2017-03-31
10277,2017-05-31,MCHP,-205.207678,2017-05-30,2017-03-31
10278,2017-06-01,MCHP,-205.207678,2017-05-30,2017-03-31
10279,2017-06-02,MCHP,-205.207678,2017-05-30,2017-03-31
10280,2017-06-05,MCHP,-205.207678,2017-05-30,2017-03-31
10281,2017-06-06,MCHP,-205.207678,2017-05-30,2017-03-31
10282,2017-06-07,MCHP,-205.207678,2017-05-30,2017-03-31
10283,2017-06-08,MCHP,-205.207678,2017-05-30,2017-03-31
10284,2017-06-09,MCHP,-205.207678,2017-05-30,2017-03-31
10285,2017-06-12,MCHP,-205.207678,2017-05-30,2017-03-31


## Validate date range, tickers, and all 43 features

In [ ]:
EXPECTED_START_DATE = pd.Timestamp('2016-07-01')
EXPECTED_END_DATE = pd.Timestamp('2026-06-30')

actual_start_date = df['date'].min()
actual_end_date = df['date'].max()

print('Expected date range:', EXPECTED_START_DATE.date(), 'to', EXPECTED_END_DATE.date())
print('Actual date range:  ', actual_start_date.date(), 'to', actual_end_date.date())

if actual_start_date != EXPECTED_START_DATE:
    raise ValueError(f'Incorrect start date: {actual_start_date.date()}')

if actual_end_date != EXPECTED_END_DATE:
    raise ValueError(f'Incorrect end date: {actual_end_date.date()}')

ticker_counts = df.groupby('ticker').size().sort_index()
display(ticker_counts.to_frame('rows'))

missing_feature_columns = [c for c in FEATURE_COLUMNS if c not in df.columns]
if missing_feature_columns:
    raise ValueError(f'Missing feature columns: {missing_feature_columns}')

feature_missing_counts = df[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False)
display(feature_missing_counts[feature_missing_counts > 0].to_frame('missing'))

if df[FEATURE_COLUMNS].isna().any().any():
    raise ValueError('Some of the 43 features still contain missing values.')

if np.isinf(df[FEATURE_COLUMNS].to_numpy(dtype=float)).any():
    raise ValueError('Infinite values remain in the 43 features.')

print('All 43 features are present and populated.')


Expected date range: 2016-07-01 to 2026-06-30
Actual date range:   2016-07-01 to 2026-06-30


,rows
ticker,
ADI,2512
AMD,2512
AVGO,2512
INTC,2512
MCHP,2512
MPWR,2512
MRVL,2512
MU,2512
NVDA,2512


,missing


All 43 features are present and populated.


## Verify that backtracked features exist on July 1, 2016

In [ ]:
BEGINNING_FEATURES = [
    'daily_return',
    'previous_day_return',
    'previous_5_day_return',
    'previous_20_day_return',
    'previous_60_day_return',
    'sp500_return',
    'nasdaq_return',
    'soxx_return',
    'qqq_return',
]

first_date_rows = df.loc[
    df['date'] == EXPECTED_START_DATE,
    ['date', 'ticker'] + BEGINNING_FEATURES,
]

display(first_date_rows)

first_date_missing = first_date_rows[BEGINNING_FEATURES].isna().sum()
if first_date_missing.sum() > 0:
    raise ValueError(
        'Beginning features are still missing:\n'
        f'{first_date_missing[first_date_missing > 0]}'
    )

print('All backtracked features are populated on July 1, 2016.')


,date,ticker,daily_return,previous_day_return,previous_5_day_return,previous_20_day_return,previous_60_day_return,sp500_return,nasdaq_return,soxx_return,qqq_return
0,2016-07-01,ADI,-0.000176,0.018339,-0.025297,-0.028140,-0.035640,0.001949,0.004109,-0.008223,0.005022
2512,2016-07-01,AMD,-0.013619,0.001949,-0.013436,0.212264,0.835714,0.001949,0.004109,-0.008223,0.005022
5024,2016-07-01,AVGO,-0.007593,0.002581,-0.020300,0.006328,-0.010472,0.001949,0.004109,-0.008223,0.005022
7536,2016-07-01,INTC,-0.001524,0.027247,-0.005759,0.032746,0.031276,0.001949,0.004109,-0.008223,0.005022
10048,2016-07-01,MCHP,-0.006698,0.021328,-0.039364,-0.018562,0.051417,0.001949,0.004109,-0.008223,0.005022
12560,2016-07-01,MPWR,-0.010539,0.020921,-0.019365,-0.002234,0.072728,0.001949,0.004109,-0.008223,0.005022
15072,2016-07-01,MRVL,0.000000,0.009534,-0.084534,-0.067504,-0.121351,0.001949,0.004109,-0.008223,0.005022
17584,2016-07-01,MU,-0.091570,0.043215,-0.020641,0.049581,0.314231,0.001949,0.004109,-0.008223,0.005022
20096,2016-07-01,NVDA,-0.007445,0.007717,-0.030522,-0.002758,0.316538,0.001949,0.004109,-0.008223,0.005022
22608,2016-07-01,ON,-0.004535,0.014960,-0.106383,-0.105477,-0.095385,0.001949,0.004109,-0.008223,0.005022


All backtracked features are populated on July 1, 2016.


## Save the cleaned master dataset

In [ ]:
final_columns = (
    ['date', 'ticker']
    + FEATURE_COLUMNS
    + TARGET_COLUMNS + ['filed_date', 'period_end']
)

df = df[final_columns].copy()

assert df.shape[1] == 50, (
    f'Expected 48 total columns, but found {df.shape[1]}.'
)

df.to_csv(OUTPUT_FILE, index=False)

print('Cleaned dataset saved to:')
print(OUTPUT_FILE)
print('Final dataset shape:', df.shape)


Cleaned dataset saved to:
/content/drive/MyDrive/Colab Notebooks/CS6140-Final-Project/processed_data2/semiconductor_model_data_43_features_cleaned.csv
Final dataset shape: (30144, 50)


## Create the supervised training dataset

Rows with unavailable future targets remain in the master file but are excluded from supervised training.

In [ ]:
master_data = df.copy()

# Remove rows without a known 21-trading-day future target.
# Then sort globally by date so the panel is chronological.
training_data = (
    master_data
    .dropna(subset=TARGET_COLUMNS)
    .sort_values(['date', 'ticker'])
    .reset_index(drop=True)
)

X = training_data[FEATURE_COLUMNS].copy()
y = training_data[TRAINING_TARGET].copy()
dates = training_data['date'].copy()
tickers = training_data['ticker'].copy()

print('Master dataset shape:', master_data.shape)
print('Training dataset shape:', training_data.shape)
print('X shape:', X.shape)
print('y shape:', y.shape)
print(
    'Rows excluded because the future target is unavailable:',
    len(master_data) - len(training_data),
)
print('Training date range:', training_data['date'].min(), 'to', training_data['date'].max())
print('Training data sorted by date:', training_data['date'].is_monotonic_increasing)


Master dataset shape: (30144, 50)
Training dataset shape: (30096, 50)
X shape: (30096, 43)
y shape: (30096,)
Rows excluded because the future target is unavailable: 48
Training date range: 2016-07-01 00:00:00 to 2026-06-24 00:00:00
Training data sorted by date: True


## Leakage-safe quantile clipping transformer

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        columns=None,
        lower_quantile=0.01,
        upper_quantile=0.99,
    ):
        self.columns = columns
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):
        X = self._to_dataframe(X)
        self.columns_ = list(X.columns) if self.columns is None else list(self.columns)
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

        for column in self.columns_:
            self.lower_bounds_[column] = X[column].quantile(self.lower_quantile)
            self.upper_bounds_[column] = X[column].quantile(self.upper_quantile)

        return self

    def transform(self, X):
        X = self._to_dataframe(X).copy()

        for column in self.columns_:
            X[column] = X[column].clip(
                lower=self.lower_bounds_[column],
                upper=self.upper_bounds_[column],
            )

        return X

    @staticmethod
    def _to_dataframe(X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError('QuantileClipper expects a pandas DataFrame.')
        return X


## Model pipelines

In [ ]:
!pip -q install xgboost

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


linear_regression_pipeline = Pipeline(
    steps=[
        (
            'clip_outliers',
            QuantileClipper(columns=['debt_to_equity']),
        ),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LinearRegression()),
    ]
)

random_forest_pipeline = Pipeline(
    steps=[
        (
            'clip_outliers',
            QuantileClipper(columns=['debt_to_equity']),
        ),
        ('imputer', SimpleImputer(strategy='median')),
        (
            'model',
            RandomForestRegressor(
                n_estimators=300,
                max_depth=None,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgboost_pipeline = Pipeline(
    steps=[
        (
            'clip_outliers',
            QuantileClipper(columns=['debt_to_equity']),
        ),
        ('imputer', SimpleImputer(strategy='median')),
        (
            'model',
            XGBRegressor(
                objective='reg:squarederror',
                n_estimators=500,
                learning_rate=0.03,
                max_depth=4,
                min_child_weight=5,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

print('All three model pipelines are ready.')


All three model pipelines are ready.


## Leakage-safe panel TimeSeriesSplit by unique dates

This split keeps all tickers from the same trading date together. It avoids the error of splitting the ticker-grouped row order directly.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Split the panel using unique trading dates rather than individual rows.
unique_dates = np.array(sorted(training_data['date'].unique()))

time_series_cv = TimeSeriesSplit(n_splits=5)
date_splits = []

for fold_number, (train_date_idx, test_date_idx) in enumerate(
    time_series_cv.split(unique_dates),
    start=1,
):
    train_dates = unique_dates[train_date_idx]
    test_dates = unique_dates[test_date_idx]

    train_mask = training_data['date'].isin(train_dates)
    test_mask = training_data['date'].isin(test_dates)

    train_indices = training_data.index[train_mask].to_numpy()
    test_indices = training_data.index[test_mask].to_numpy()

    date_splits.append((train_indices, test_indices))

    print(
        f'Fold {fold_number}: '
        f'train {pd.Timestamp(train_dates.min()).date()} to '
        f'{pd.Timestamp(train_dates.max()).date()} '
        f'({len(train_indices):,} rows), '
        f'test {pd.Timestamp(test_dates.min()).date()} to '
        f'{pd.Timestamp(test_dates.max()).date()} '
        f'({len(test_indices):,} rows)'
    )

    # Confirm that no trading date appears in both sets.
    overlap = set(train_dates).intersection(set(test_dates))
    assert not overlap, f'Date leakage found in fold {fold_number}: {overlap}'

print('\nDate-based cross-validation splits created:', len(date_splits))


Fold 1: train 2016-07-01 to 2018-02-28 (5,016 rows), test 2018-03-01 to 2019-10-25 (5,016 rows)
Fold 2: train 2016-07-01 to 2019-10-25 (10,032 rows), test 2019-10-28 to 2021-06-24 (5,016 rows)
Fold 3: train 2016-07-01 to 2021-06-24 (15,048 rows), test 2021-06-25 to 2023-02-22 (5,016 rows)
Fold 4: train 2016-07-01 to 2023-02-22 (20,064 rows), test 2023-02-23 to 2024-10-21 (5,016 rows)
Fold 5: train 2016-07-01 to 2024-10-21 (25,080 rows), test 2024-10-22 to 2026-06-24 (5,016 rows)

Date-based cross-validation splits created: 5


## Example: evaluate a pipeline with the date-based folds

The same `date_splits` object can be reused for Linear Regression, Random Forest, and XGBoost.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_pipeline_with_date_splits(model_pipeline, X, y, splits, model_name):
    results = []

    for fold_number, (train_indices, test_indices) in enumerate(splits, start=1):
        X_train = X.loc[train_indices].copy()
        y_train = y.loc[train_indices].copy()
        X_test = X.loc[test_indices].copy()
        y_test = y.loc[test_indices].copy()

        model_pipeline.fit(X_train, y_train)
        predictions = model_pipeline.predict(X_test)

        rmse = mean_squared_error(y_test, predictions) ** 0.5
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)

        results.append({
            'Model': model_name,
            'Fold': fold_number,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2,
        })

    return pd.DataFrame(results)


# Example usage:
# linear_results = evaluate_pipeline_with_date_splits(
#     linear_regression_pipeline,
#     X,
#     y,
#     date_splits,
#     'Linear Regression',
# )
# display(linear_results)
